# 07 — Explainability and Error Analysis

**Objective** — inspect where the selected models fail and which features most affect predictive performance.

**Input** — final grouped train/test split and selected regression/classification models.

**Output** — error tables and permutation-importance summaries for both tasks.

## 1. Rebuild the Final Evaluation Split

In [ ]:
from pathlib import Path
import sys
import pandas as pd

current_directory = Path.cwd().resolve()
project_root = next(
    directory for directory in [current_directory, *current_directory.parents]
    if (directory / 'src').is_dir() and (directory / 'notebooks').is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features import prepare_feature_frame, build_preprocessor
from src.modeling import grouped_train_test_split, derive_high_cost_labels, get_regression_candidates, get_classification_candidates
from src.evaluation import regression_error_table, classification_error_table, permutation_importance_table

processed_data_path = project_root / 'data' / 'processed' / 'analytical_dataset.csv'
analytical_dataset = pd.read_csv(processed_data_path)
features, target_charges, customer_groups = prepare_feature_frame(analytical_dataset)
training_features, test_features, training_charges, test_charges, training_groups, test_groups = grouped_train_test_split(
    features, target_charges, customer_groups
)
preprocessor = build_preprocessor(training_features)

## 2. Regression Error Analysis

In [ ]:
selected_regression_model = get_regression_candidates(preprocessor)['Random Forest'].fit(
    training_features, training_charges
)
regression_predictions = selected_regression_model.predict(test_features)
regression_error_table(
    pd.DataFrame({'Customer ID': test_groups.values}),
    test_charges.reset_index(drop=True),
    regression_predictions,
).head(10)

### Regression Permutation Importance

In [ ]:
permutation_importance_table(
    selected_regression_model, test_features, test_charges,
    'neg_mean_absolute_error', n_repeats=5
).head(10)

## 3. Classification Error Analysis

In [ ]:
training_high_cost_labels, test_high_cost_labels, high_cost_threshold = derive_high_cost_labels(
    training_charges, test_charges
)
selected_classifier = get_classification_candidates(preprocessor)['HistGradientBoosting'].fit(
    training_features, training_high_cost_labels
)
classification_probability_scores = selected_classifier.predict_proba(test_features)[:, 1]
classification_predictions = (classification_probability_scores >= 0.5).astype(int)
classification_error_table(
    pd.DataFrame({'Customer ID': test_groups.values}),
    test_high_cost_labels.reset_index(drop=True),
    classification_predictions,
    classification_probability_scores,
).query("error_type != 'correct'").head(10)

### Classification Permutation Importance

In [ ]:
permutation_importance_table(
    selected_classifier, test_features, test_high_cost_labels,
    'average_precision', n_repeats=5
).head(10)

## Takeaways

Error analysis shows model limits that aggregate metrics can hide. Permutation importance shows how much predictive performance depends on each feature. It does not prove medical causality.